# Análisis del Dataset PROFE 2025
### IC-UNED Spanish Reading Comprehension

Este notebook analiza el subconjunto de datos procesados para el desarrollo del proyecto PROFE 2025, compuesto por preguntas de comprensión lectora de tipo opción múltiple extraídas de exámenes del Instituto Cervantes.

**Ficheros utilizados:**
- `filtered_multiple_choice.json` — dataset original con textos, preguntas y opciones
- `subset_100.json` — subconjunto completo de respuestas (135 preguntas, incluyendo imágenes)
- `subset_images.json` — subconjunto con solo los ejercicios que contienen imágenes (35 preguntas)

---
## 0. Imports y carga de datos

In [2]:
import json
from collections import defaultdict, Counter
import pprint

# --- Carga de ficheros ---
with open('../data/filtered_multiple_choice.json', 'r', encoding='utf-8') as f:
    dataset = json.load(f)

with open('../data/subset_100.json', 'r', encoding='utf-8') as f:
    subset_100 = json.load(f)

with open('../data/subset_images.json', 'r', encoding='utf-8') as f:
    subset_images = json.load(f)

exams = dataset['exams']

print(f"Dataset cargado: '{dataset['dataset-name']}'")
print(f"Licencia: {dataset['dataset-license']}")
print(f"Número de exámenes: {len(exams)}")
print(f"Respuestas en subset_100:   {len(subset_100)}")
print(f"Respuestas en subset_images: {len(subset_images)}")

Dataset cargado: 'IC-UNED Spanish Reading Comprehension'
Licencia: CC BY-NC-SA 4.0
Número de exámenes: 25
Respuestas en subset_100:   135
Respuestas en subset_images: 35


---
## 1. Descripción del dataset

In [3]:
print("=== Descripción (ES) ===")
print(dataset['dataset-desc-es'])
print()
print("=== Description (EN) ===")
print(dataset['dataset-desc-en'])
print()
print("Contacto:", dataset['contact'])
print("Funding:",  dataset['funding'])

=== Descripción (ES) ===
Corpus de las tareas de comprensión lectora contenidas en los exámenes creados por el Instituto Cervantes para la evaluación de estudiantes de español en diferentes niveles

=== Description (EN) ===
Corpus of the reading comprehension tasks contained in the exams created by the Cervantes Institute for the evaluation of Spanish students at different levels

Contacto: anselmo@lsi.uned.es
Funding: DeepInfo Project (AEI PID2021-127777OB-C22), UNED, Spain


---
## 2. Estructura de un examen

Cada entrada en `exams` sigue esta jerarquía:

```
exam
 ├── level        (A1, A2, B1, B2)
 ├── examId       (e.g. "A1_2010-11-19")
 └── exercises[]
      ├── exerciseID
      ├── type            (siempre "multiple-choice" en este subconjunto)
      ├── instructions    (texto de instrucciones)
      └── exercise
           ├── text             (texto de lectura, puede estar vacío si hay imagen)
           ├── image-path       (ruta relativa a imagen del enunciado, si procede)
           └── questions[]
                ├── questionId
                ├── text
                ├── image-path  (imagen de la pregunta, si procede)
                └── options[]
                     ├── optionId    (A, B, C, D)
                     ├── text
                     └── image-path  (imagen de la opción, si procede)
```

In [4]:
# Ejemplo: primer examen del dataset
ejemplo = exams[0]
print(f"examId : {ejemplo['examId']}")
print(f"level  : {ejemplo['level']}")
print(f"Número de ejercicios: {len(ejemplo['exercises'])}")

ex = ejemplo['exercises'][0]
print(f"\nexerciseID : {ex['exerciseID']}")
print(f"type       : {ex['type']}")
print(f"Instrucciones:\n  {ex['instructions'][:120]}...")

ex_data = ex['exercise']
print(f"\nTexto (primeros 300 chars):\n  {ex_data['text'][:300]}...")
print(f"\nNúmero de preguntas: {len(ex_data['questions'])}")

q = ex_data['questions'][0]
print(f"\nEjemplo pregunta: [{q['questionId']}]")
print(f"  Texto: {q['text']}")
for opt in q['options']:
    print(f"  ({opt['optionId']}) {opt['text']}")

examId : A1_2010-11-19
level  : A1
Número de ejercicios: 1

exerciseID : A1_2010-11-19_E1
type       : multiple-choice
Instrucciones:
  Lea esta carta. A continuación hay 5 preguntas sobre ella. Usted debe seleccionar la
opción correcta (A, B, C o D). 
Deb...

Texto (primeros 300 chars):
  Duerida ALicia -
¿Qué tal estás? ¿Te gusta tu nuevo trabajo en Barcelona? Yo en Madrid
estoy muy bien. Tengo que darte buenas noticias. La próxima semana voy a
Barcelona con mi hija Celia, ¿ la recuerdas? Va a hacer este verano un curso
sobre cine allí. El curso es de un mes y empieza el 15 de agost...

Número de preguntas: 4

Ejemplo pregunta: [A1_2010-11-19_E1_Q1]
  Texto: Marta va a viajar…
  (A) por trabajo.
  (B) a Madrid.
  (C) para estudiar.
  (D) a Barcelona.


---
## 3. Estadísticas globales del dataset

In [5]:
# Construir tabla resumen por nivel
stats = defaultdict(lambda: {
    'exams': 0, 'exercises': 0, 'questions': 0,
    'exercises_with_images': 0, 'questions_with_img_options': 0
})

for exam in exams:
    lvl = exam['level']
    stats[lvl]['exams'] += 1
    for ex in exam['exercises']:
        stats[lvl]['exercises'] += 1
        ex_data = ex['exercise']
        qs = ex_data.get('questions', [])
        stats[lvl]['questions'] += len(qs)
        # Ejercicio con imagen de enunciado
        if ex_data.get('image-path', ''):
            stats[lvl]['exercises_with_images'] += 1
        # Preguntas con imagen en opciones
        for q in qs:
            if any(opt.get('image-path', '') for opt in q.get('options', [])):
                stats[lvl]['questions_with_img_options'] += 1

print(f"{'Nivel':<8} {'Exámenes':>9} {'Ejercicios':>11} {'Preguntas':>10} {'Ej. c/img':>10} {'Q c/img opts':>13}")
print("-" * 60)
totals = defaultdict(int)
for lvl in sorted(stats.keys()):
    s = stats[lvl]
    print(f"{lvl:<8} {s['exams']:>9} {s['exercises']:>11} {s['questions']:>10} "
          f"{s['exercises_with_images']:>10} {s['questions_with_img_options']:>13}")
    for k in s: totals[k] += s[k]
print("-" * 60)
print(f"{'TOTAL':<8} {totals['exams']:>9} {totals['exercises']:>11} {totals['questions']:>10} "
      f"{totals['exercises_with_images']:>10} {totals['questions_with_img_options']:>13}")

Nivel     Exámenes  Ejercicios  Preguntas  Ej. c/img  Q c/img opts
------------------------------------------------------------
A1              16          16         71          0             7
A2               3          11         23          0             0
B1               4           4         21          0             0
B2               2           7         20          0             0
------------------------------------------------------------
TOTAL           25          38        135          0             7


---
## 4. Distribución de respuestas (subset_100)

In [6]:
# Mapear questionId -> nivel
q_level = {}
q_to_exam = {}
for exam in exams:
    for ex in exam['exercises']:
        for q in ex['exercise'].get('questions', []):
            q_level[q['questionId']] = exam['level']
            q_to_exam[q['questionId']] = exam['examId']

# Distribución de respuestas por nivel
level_answers = defaultdict(list)
for qid, ans in subset_100.items():
    lvl = q_level.get(qid, '?')
    level_answers[lvl].append(ans)

print("Distribución de respuestas por nivel (subset_100)")
print(f"{'Nivel':<8} {'A':>5} {'B':>5} {'C':>5} {'D':>5} {'Total':>7}")
print("-" * 38)
grand = Counter()
for lvl in sorted(level_answers.keys()):
    ctr = Counter(level_answers[lvl])
    grand.update(ctr)
    print(f"{lvl:<8} {ctr.get('A',0):>5} {ctr.get('B',0):>5} {ctr.get('C',0):>5} {ctr.get('D',0):>5} {sum(ctr.values()):>7}")
print("-" * 38)
print(f"{'TOTAL':<8} {grand.get('A',0):>5} {grand.get('B',0):>5} {grand.get('C',0):>5} {grand.get('D',0):>5} {sum(grand.values()):>7}")

print()
print("Nota: D solo aparece en A1 porque A2/B1/B2 usan preguntas de 3 opciones (A/B/C).")

Distribución de respuestas por nivel (subset_100)
Nivel        A     B     C     D   Total
--------------------------------------
A1          17    26    19     9      71
A2           8     4    11     0      23
B1           7     7     7     0      21
B2          11     6     3     0      20
--------------------------------------
TOTAL       43    43    40     9     135

Nota: D solo aparece en A1 porque A2/B1/B2 usan preguntas de 3 opciones (A/B/C).


---
## 5. Distribución de opciones por nivel (¿cuántas opciones por pregunta?)

In [7]:
# Número de opciones por pregunta, por nivel
option_dist = defaultdict(Counter)
for exam in exams:
    lvl = exam['level']
    for ex in exam['exercises']:
        for q in ex['exercise'].get('questions', []):
            n_opts = len(q.get('options', []))
            option_dist[lvl][n_opts] += 1

print("Número de opciones por pregunta, agrupado por nivel:")
print(f"{'Nivel':<8} {'2 opts':>8} {'3 opts':>8} {'4 opts':>8}")
print("-" * 38)
for lvl in sorted(option_dist.keys()):
    c = option_dist[lvl]
    print(f"{lvl:<8} {c.get(2,0):>8} {c.get(3,0):>8} {c.get(4,0):>8}")

Número de opciones por pregunta, agrupado por nivel:
Nivel      2 opts   3 opts   4 opts
--------------------------------------
A1              0       20       51
A2              0       23        0
B1              0       21        0
B2              3       17        0


---
## 6. Desglose por examen

In [8]:
print(f"{'examId':<30} {'Nivel':<6} {'Ejercicios':>11} {'Preguntas':>10} {'Tipo ID':>20}")
print("-" * 80)
for exam in exams:
    n_ex = len(exam['exercises'])
    n_q  = sum(len(ex['exercise'].get('questions',[])) for ex in exam['exercises'])
    # Identificar si es examen oficial (fecha) o muestra (Sample)
    tipo = 'Sample' if 'Sample' in exam['examId'] else 'Oficial'
    print(f"{exam['examId']:<30} {exam['level']:<6} {n_ex:>11} {n_q:>10} {tipo:>20}")

examId                         Nivel   Ejercicios  Preguntas              Tipo ID
--------------------------------------------------------------------------------
A1_2010-11-19                  A1               1          4              Oficial
A1_Sample_Ines                 A1               1          5               Sample
A1_Sample_Carol                A1               1          5               Sample
A1_Sample_Fran                 A1               1          5               Sample
A1_Sample_Kaki                 A1               1          5               Sample
A1_Sample_Susana               A1               1          5               Sample
A1_Sample_Pierre               A1               1          5               Sample
A1_2010-11-20                  A1               1          4              Oficial
A1_2011-11-18                  A1               1          4              Oficial
A1_2011-11-19                  A1               1          4              Oficial
A1_2012-05-25    

---
## 7. Análisis de ejercicios con imágenes

In [9]:
def get_image_paths(obj, results=None, path=''):
    """Extrae recursivamente todos los image-path no vacíos."""
    if results is None:
        results = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k == 'image-path' and v:
                results.append((path, v))
            get_image_paths(v, results, f"{path}.{k}")
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            get_image_paths(v, results, f"{path}[{i}]")
    return results

all_img_paths = get_image_paths(exams)
print(f"Total de image-path no vacíos en el dataset: {len(all_img_paths)}")
print()

# Ejercicios con imágenes, agrupados por examId
img_by_exam = defaultdict(list)
for exam in exams:
    for ex in exam['exercises']:
        ex_data = ex['exercise']
        img_paths_ex = get_image_paths(ex_data)
        if img_paths_ex:
            img_by_exam[exam['examId']].extend([v for _, v in img_paths_ex])

print(f"{'examId':<30} {'Nivel':<6} {'Rutas de imagen':>15}")
print("-" * 65)
for exam in exams:
    eid = exam['examId']
    if eid in img_by_exam:
        paths = img_by_exam[eid]
        print(f"{eid:<30} {exam['level']:<6} {len(paths):>15}")
        for p in paths[:3]:
            print(f"  {'':30} {'':6}   → {p}")
        if len(paths) > 3:
            print(f"  {'':30} {'':6}   ... y {len(paths)-3} más")

Total de image-path no vacíos en el dataset: 24

examId                         Nivel  Rutas de imagen
-----------------------------------------------------------------
A1_Sample_Ines                 A1                   3
                                          → img/A1_Sample_Ines_E1_Q5_A.png
                                          → img/A1_Sample_Ines_E1_Q5_B.png
                                          → img/A1_Sample_Ines_E1_Q5_C.png
A1_Sample_Carol                A1                   3
                                          → img/A1_Sample_Carol_E1_Q5_A.png
                                          → img/A1_Sample_Carol_E1_Q5_B.png
                                          → img/A1_Sample_Carol_E1_Q5_C.png
A1_Sample_Fran                 A1                   4
                                          → img/A1_Sample_Fran_E1_Q5_A.png
                                          → img/A1_Sample_Fran_E1_Q5_B.png
                                          → img/A1_Sample_Fran_E1_

---
## 8. Subconjunto de imágenes (subset_images)

`subset_images.json` contiene exactamente los exámenes tipo **Sample** (examinandos con nombres propios) y el examen oficial **A1_2020-07-01**, que son los que incluyen referencias a ficheros de imagen en sus opciones.

In [10]:
# Identificar los examIds presentes en subset_images
img_exam_ids = set()
for qid in subset_images.keys():
    img_exam_ids.add(q_to_exam.get(qid, '?'))

print(f"subset_images cubre {len(subset_images)} preguntas de {len(img_exam_ids)} exámenes:")
for eid in sorted(img_exam_ids):
    qs_in_exam = [qid for qid in subset_images if q_to_exam.get(qid) == eid]
    lvl = q_level.get(qs_in_exam[0], '?')
    print(f"  {eid:<30} nivel={lvl}  preguntas={len(qs_in_exam)}")

print()
# Verificar que subset_images ⊆ subset_100
overlap = set(subset_images.keys()) & set(subset_100.keys())
print(f"Preguntas de subset_images que también están en subset_100: {len(overlap)} / {len(subset_images)}")
consistent = all(subset_images[k] == subset_100[k] for k in overlap)
print(f"Respuestas consistentes entre ambos subsets: {consistent}")

subset_images cubre 35 preguntas de 7 exámenes:
  A1_2020-07-01                  nivel=A1  preguntas=5
  A1_Sample_Carol                nivel=A1  preguntas=5
  A1_Sample_Fran                 nivel=A1  preguntas=5
  A1_Sample_Ines                 nivel=A1  preguntas=5
  A1_Sample_Kaki                 nivel=A1  preguntas=5
  A1_Sample_Pierre               nivel=A1  preguntas=5
  A1_Sample_Susana               nivel=A1  preguntas=5

Preguntas de subset_images que también están en subset_100: 35 / 35
Respuestas consistentes entre ambos subsets: True


---
## 9. Correspondencia entre subset_100 y el dataset original

In [11]:
# Todos los questionId del dataset
all_q_ids = set()
for exam in exams:
    for ex in exam['exercises']:
        for q in ex['exercise'].get('questions', []):
            all_q_ids.add(q['questionId'])

subset_keys = set(subset_100.keys())

print(f"Total questionIds en dataset original : {len(all_q_ids)}")
print(f"Total claves en subset_100           : {len(subset_keys)}")
print(f"Cobertura (subset_100 ∩ dataset)     : {len(subset_keys & all_q_ids)}")
missing_in_dataset = subset_keys - all_q_ids
missing_in_subset  = all_q_ids - subset_keys
print(f"Claves en subset_100 no encontradas en dataset : {len(missing_in_dataset)}")
print(f"Preguntas del dataset no cubiertas por subset_100: {len(missing_in_subset)}")
if missing_in_dataset:
    print("  Claves huérfanas:", missing_in_dataset)
if missing_in_subset:
    print("  Preguntas sin respuesta manual:", missing_in_subset)

Total questionIds en dataset original : 135
Total claves en subset_100           : 135
Cobertura (subset_100 ∩ dataset)     : 135
Claves en subset_100 no encontradas en dataset : 0
Preguntas del dataset no cubiertas por subset_100: 0


---
## 10. Vista detallada por nivel — A1

In [12]:
def mostrar_ejercicio(exam, max_preguntas=2):
    """Imprime un resumen legible de un examen."""
    print(f"{'='*60}")
    print(f"ExamId : {exam['examId']}   Nivel: {exam['level']}")
    for ex in exam['exercises']:
        ex_data = ex['exercise']
        print(f"\nEjercicio: {ex['exerciseID']}")
        texto = ex_data.get('text', '').strip()
        if texto:
            print(f"Texto (extracto): {texto[:200]}...")
        elif ex_data.get('image-path', ''):
            print(f"[Imagen del enunciado: {ex_data['image-path']}]")
        print(f"Preguntas ({len(ex_data.get('questions',[]))} total):")
        for q in ex_data.get('questions', [])[:max_preguntas]:
            ans = subset_100.get(q['questionId'], 'N/A')
            img_opts = [opt.get('image-path','') for opt in q.get('options',[])]
            has_img = any(img_opts)
            print(f"  [{q['questionId']}]  {q['text']}")
            for opt in q['options']:
                marker = ' ✓' if opt['optionId'] == ans else ''
                if opt.get('image-path', ''):
                    print(f"    ({opt['optionId']}) [imagen: {opt['image-path']}]{marker}")
                else:
                    print(f"    ({opt['optionId']}) {opt['text']}{marker}")
        if len(ex_data.get('questions',[])) > max_preguntas:
            print(f"  ... ({len(ex_data['questions'])-max_preguntas} preguntas más)")

# Mostrar el primer examen A1 oficial
a1_exams = [e for e in exams if e['level'] == 'A1' and 'Sample' not in e['examId']]
mostrar_ejercicio(a1_exams[0])

ExamId : A1_2010-11-19   Nivel: A1

Ejercicio: A1_2010-11-19_E1
Texto (extracto): Duerida ALicia -
¿Qué tal estás? ¿Te gusta tu nuevo trabajo en Barcelona? Yo en Madrid
estoy muy bien. Tengo que darte buenas noticias. La próxima semana voy a
Barcelona con mi hija Celia, ¿ la recuer...
Preguntas (4 total):
  [A1_2010-11-19_E1_Q1]  Marta va a viajar…
    (A) por trabajo.
    (B) a Madrid.
    (C) para estudiar.
    (D) a Barcelona. ✓
  [A1_2010-11-19_E1_Q2]  El curso…
    (A) dura una semana.
    (B) es de cine ✓
    (C) dura 15 días.
    (D) es de arte.
  ... (2 preguntas más)


---
## 11. Vista detallada por nivel — A2

In [13]:
a2_exams = [e for e in exams if e['level'] == 'A2']
# A2 tiene múltiples ejercicios por examen — mostrar el primero
mostrar_ejercicio(a2_exams[0], max_preguntas=2)

ExamId : A2_2010-05-22   Nivel: A2

Ejercicio: A2_2010-05-22_E2
Texto (extracto): Para: Ángela
Asunto: RE: ¿Qué tal?
Ya veo que no has parado en toda la semana. ¡Qué bien vives!
Yo también he estado fuera un par de días y tengo una propuesta. ¿Te apetece venir a la final de la Copa...
Preguntas (4 total):
  [A2_2010-05-22_E2_Q8]  Beatriz cuenta lo que hizo…
    (A) el fin de semana. ✓
    (B) la semana pasada.
    (C) en vacaciones.
  [A2_2010-05-22_E2_Q9]  Beatriz fue a Bilbao para…
    (A) salir en la televisión.
    (B) jugar un partido.
    (C) ver y animar a su equipo. ✓
  ... (2 preguntas más)

Ejercicio: A2_2010-05-22_E30.0
Texto (extracto): ¿Te gusta hacer deporte? Apúntate al nuevo equipo de fútbol femenino de la universidad. Necesitamos jugadoras de 18 a 25 años para todas las posiciones. Las pruebas de selección son este sábado a las ...
Preguntas (1 total):
  [A2_2010-05-22_E3_Q0]  En este anuncio buscan…
    (A) chicas para jugar al fútbol. ✓
    (B) un lugar donde hacer d

---
## 12. Vista detallada por nivel — B1

In [14]:
b1_exams = [e for e in exams if e['level'] == 'B1']
mostrar_ejercicio(b1_exams[0], max_preguntas=2)

ExamId : B1_2015-07-01   Nivel: B1

Ejercicio: B1_2015-07-01_E2
Texto (extracto): México, campeón mundial de Scrabble en español
Jesús Ortega Calzada, de 35 años y originario del Distrito Federal, ha sido el primer mexicano campeón Mundial de Scrabble en español. El Scrabble es un ...
Preguntas (6 total):
  [B1_2015-07-01_E2_Q7]  Jesús descubrió este juego porque…
    (A) lo aprendió en la escuela.
    (B) un amigo le enseñó a jugar. ✓
    (C) era muy parecido a otros juegos.
  [B1_2015-07-01_E2_Q8]  Según el texto, en este juego…
    (A) la concentración es la clave.
    (B) también influye la suerte. ✓
    (C) lo fundamental es practicar.
  ... (4 preguntas más)


---
## 13. Vista detallada por nivel — B2

In [15]:
b2_exams = [e for e in exams if e['level'] == 'B2']
mostrar_ejercicio(b2_exams[0], max_preguntas=2)

ExamId : B2_2005-11-18   Nivel: B2

Ejercicio: B2_2005-11-18_E1B
Texto (extracto): INÉDITO DESCUBRIMIENTO
Un hecho sin precedentes se suscitó hace un par de semanas al noroeste de la isla Grande de Chiloé, cuando unos investigadores vieron un grupo de veintitrés ballenas azules. En ...
Preguntas (2 total):
  [B2_2005-11-18_E1_Q5]  El descubrimiento en Chile es importante porque las ballenas:
    (A) estaban cerca de la costa. ✓
    (B) eran jóvenes.
    (C) tenían un gran tamaño.
  [B2_2005-11-18_E1_Q6]  En el texto se aﬁrma que el proyecto para conservar las ballenas:
    (A) lo patrocina Directemar. ✓
    (B) lo presentaron dos organismos.
    (C) todavía no se ha presentado.

Ejercicio: B2_2005-11-18_E1C
Texto (extracto): EJECUTIVOS A LA MODA
A primera vista, la moda puede parecer una manifestación individual de vanidad; pero también reﬂeja la cultura, los vaivenes socioeconómicos y los estilos de vida de un determinad...
Preguntas (3 total):
  [B2_2005-11-18_E1_Q7]  Según Carola Ca

---
## 14. Exámenes Sample con imágenes en opciones

In [16]:
sample_exams = [e for e in exams if 'Sample' in e['examId']]
print(f"Exámenes tipo Sample: {len(sample_exams)}")
print()
# Mostrar uno completo
mostrar_ejercicio(sample_exams[0], max_preguntas=5)

Exámenes tipo Sample: 6

ExamId : A1_Sample_Ines   Nivel: A1

Ejercicio: A1_Sample_Ines_E1
Texto (extracto): De: ines@mail.com
Para: pedroizquierdo@hotmail.com
Noticias

Hola, Pedro:
¿Qué tal estás? ¿Tienes muchos exámenes finales? Yo ahora estudio bastante para tener buenas notas y unas buenas vacaciones.
E...
Preguntas (5 total):
  [A1_Sample_Ines_E1_Q1]  En este correo, Inés le cuenta a Pedro...
    (A) cuándo termina los exámenes.
    (B) por qué quiere trabajar en verano. ✓
    (C) dónde va a ir de vacaciones en julio.
  [A1_Sample_Ines_E1_Q2]  En el texto se dice que...
    (A) las fiestas de San Sebastián son bonitas.
    (B) la familia de Marta tiene un hotel en Santander.
    (C) el 15 de agosto Inés va a estar en Bilbao. ✓
  [A1_Sample_Ines_E1_Q3]  Inés y Marta van a ir de Santander a Madrid...
    (A) en coche.
    (B) en moto. ✓
    (C) en autobús.
  [A1_Sample_Ines_E1_Q4]  La fiesta de cumpleaños de Inés es...
    (A) el jueves.
    (B) el viernes.
    (C) el sábado. ✓
  [

---
## 15. Función auxiliar: buscar pregunta por ID

In [17]:
# Índice rápido: questionId -> (exam, exercise, question)
q_index = {}
for exam in exams:
    for ex in exam['exercises']:
        for q in ex['exercise'].get('questions', []):
            q_index[q['questionId']] = {
                'exam': exam,
                'exercise': ex,
                'question': q
            }

def get_question(qid):
    """Devuelve un dict con toda la información de una pregunta por su ID."""
    if qid not in q_index:
        print(f"ID '{qid}' no encontrado.")
        return None
    entry = q_index[qid]
    q = entry['question']
    exam = entry['exam']
    ex_data = entry['exercise']['exercise']
    ans = subset_100.get(qid, 'N/A')
    
    print(f"QuestionId : {qid}")
    print(f"ExamId     : {exam['examId']}   Nivel: {exam['level']}")
    print(f"Texto      : {q['text']}")
    for opt in q['options']:
        marker = ' ✓' if opt['optionId'] == ans else ''
        if opt.get('image-path',''):
            print(f"  ({opt['optionId']}) [imagen]{marker}")
        else:
            print(f"  ({opt['optionId']}) {opt['text']}{marker}")
    print(f"Respuesta correcta: {ans}")
    return entry

# Ejemplo de uso
_ = get_question('A1_2010-11-19_E1_Q1')

QuestionId : A1_2010-11-19_E1_Q1
ExamId     : A1_2010-11-19   Nivel: A1
Texto      : Marta va a viajar…
  (A) por trabajo.
  (B) a Madrid.
  (C) para estudiar.
  (D) a Barcelona. ✓
Respuesta correcta: D


In [18]:
# Otro ejemplo: pregunta con imagen en opciones
_ = get_question('A1_Sample_Ines_E1_Q5')

QuestionId : A1_Sample_Ines_E1_Q5
ExamId     : A1_Sample_Ines   Nivel: A1
Texto      : ¿Dónde es la fiesta de cumpleaños de Inés?
  (A) [imagen]
  (B) [imagen] ✓
  (C) [imagen]
Respuesta correcta: B


---
## 16. Resumen ejecutivo

In [19]:
print("=" * 60)
print(" RESUMEN EJECUTIVO — PROFE 2025 Dataset ")
print("=" * 60)
print()
print(f"Dataset         : {dataset['dataset-name']}")
print(f"Niveles MCER    : A1, A2, B1, B2")
print(f"Exámenes totales: {len(exams)}  (16 A1 / 3 A2 / 4 B1 / 2 B2)")
print(f"Ejercicios total: {sum(len(e['exercises']) for e in exams)}")
print(f"Preguntas total : {len(all_q_ids)}")
print()
print(f"subset_100     : {len(subset_100)} respuestas manuales (cubre el 100% del dataset)")
print(f"subset_images  : {len(subset_images)} respuestas (solo ejercicios con imágenes)")
print()
print("Observaciones clave:")
print("  • A1 usa 4 opciones (A/B/C/D). A2, B1 y B2 usan mayoritariamente 3 opciones (A/B/C).")
print("  • Las imágenes aparecen ÚNICAMENTE en las opciones de respuesta (no en el enunciado)")
print("    en 9 ejercicios: 6 exámenes Sample A1 + A1_2020-07-01 + B1_2004-05-15 + B2_2005-11-18_E1C")
print("  • Los exámenes Sample tienen nombres de personas (Ines, Carol, Fran, Kaki, Susana, Pierre).")
print("  • subset_images es un subconjunto estricto de subset_100 con respuestas idénticas.")
print()
print("Rutas de imagen: ficheros PNG bajo ./img/ con naming convention")
print("  {examId}_E{n}_Q{n}_{optionId}.png")
print("  Ejemplo: img/A1_Sample_Ines_E1_Q5_B.png")

 RESUMEN EJECUTIVO — PROFE 2025 Dataset 

Dataset         : IC-UNED Spanish Reading Comprehension
Niveles MCER    : A1, A2, B1, B2
Exámenes totales: 25  (16 A1 / 3 A2 / 4 B1 / 2 B2)
Ejercicios total: 38
Preguntas total : 135

subset_100     : 135 respuestas manuales (cubre el 100% del dataset)
subset_images  : 35 respuestas (solo ejercicios con imágenes)

Observaciones clave:
  • A1 usa 4 opciones (A/B/C/D). A2, B1 y B2 usan mayoritariamente 3 opciones (A/B/C).
  • Las imágenes aparecen ÚNICAMENTE en las opciones de respuesta (no en el enunciado)
    en 9 ejercicios: 6 exámenes Sample A1 + A1_2020-07-01 + B1_2004-05-15 + B2_2005-11-18_E1C
  • Los exámenes Sample tienen nombres de personas (Ines, Carol, Fran, Kaki, Susana, Pierre).
  • subset_images es un subconjunto estricto de subset_100 con respuestas idénticas.

Rutas de imagen: ficheros PNG bajo ./img/ con naming convention
  {examId}_E{n}_Q{n}_{optionId}.png
  Ejemplo: img/A1_Sample_Ines_E1_Q5_B.png
